## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

Repo already exists — pulling latest changes
fatal: unable to access 'https://github.com/Lv1g1/RecSys-Challenge-2025.git/': Could not resolve host: github.com


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
if IS_COLAB or IS_KAGGLE:
    !pip install optuna

import optuna

In [4]:
import importlib
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import ModelOptimizer

Running on local — storage at: /home/luigi/RecSys
Running on local — storage at: /home/luigi/RecSys


/home/luigi/RecSys/Challenge/hyper_tuning.py:75: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_study(self, study_name, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):
/home/luigi/RecSys/Challenge/hyper_tuning.py:107: ExperimentalWarning: WilcoxonPruner is experimental (supported from v3.6.0). The interface can change in the future.
  def create_and_optimize_study(self, study_name, objective_function, n_trials=50, direction="maximize", load_if_exists=True, pruner=optuna.pruners.WilcoxonPruner()):


## **Load Data**

In [5]:
folds = paths.load_cv_folds(k=5)

In [6]:
from Evaluation.Evaluator import EvaluatorHoldout

evaluators = [EvaluatorHoldout(URM_validation, cutoff_list=[20]) for URM_train, URM_validation in folds]

EvaluatorHoldout: Ignoring 40 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Ignoring 38 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Ignoring 45 ( 0.2%) Users that have less than 1 test interactions
EvaluatorHoldout: Ignoring 29 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Ignoring 30 ( 0.1%) Users that have less than 1 test interactions


## **Load Similarity Weights**

In [7]:
from Recommenders.SLIM.SLIMElasticNetRecommender import MultiThreadSLIM_SLIMElasticNetRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender

RP3beta = []
SLIM_pos = []
SLIM_neg = []

In [8]:
import json

def get_best_params(json_path: str) -> dict:
    with open(json_path, "r") as f:
        data = json.load(f)
        
        best_study = None
        for _, values in data.items():
            if best_study is None:
                best_study = values
                continue

            if values["best_score"] > best_study["best_score"]:
                best_study = values

    return best_study["best_params"]

In [9]:
path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/SLIMElasticNet.json")
best_slim_params = get_best_params(path)
print("Best SLIMElasticNet params:", best_slim_params)

path = os.path.join(LOCAL_REPO_PATH, f"performance_logs/RP3beta.json")
best_rp3_params = get_best_params(path)
print("Best RP3beta params:", best_rp3_params)

for fold_index, (URM_train, URM_validation) in enumerate(folds):
    print(f"FOLD {fold_index + 1}/{len(folds)}")

    # Train SLIM ElasticNet and RP3beta models
    slim_en_model = MultiThreadSLIM_SLIMElasticNetRecommender(URM_train)
    slim_en_model.fit(**best_slim_params, workers=8)
    
    rp3beta_model = RP3betaRecommender(URM_train)
    rp3beta_model.fit(**best_rp3_params)
    
    # Split positive and negative similarities
    slim_positive = slim_en_model.W_sparse.copy()
    slim_positive.data[slim_positive.data < 0] = 0
    slim_negative = slim_en_model.W_sparse.copy()
    slim_negative.data[slim_negative.data > 0] = 0
    slim_negative.data = np.abs(slim_negative.data)

    rp3beta = rp3beta_model.W_sparse.copy()

    # Normalize
    assert slim_positive.min() == slim_negative.min() == 0
    assert slim_positive.max() > 0 and slim_negative.max() > 0
    slim_positive_max = slim_positive.max()
    slim_positive.data /= slim_positive_max

    slim_negative_max = slim_negative.max()
    slim_negative.data /= slim_negative_max
    
    assert rp3beta.min() == 0
    assert rp3beta.max() > 0
    rp3beta_max = rp3beta.max()
    rp3beta.data /= rp3beta_max

    # Store
    SLIM_pos.append(slim_positive)
    SLIM_neg.append(slim_negative)
    
    RP3beta.append(rp3beta)

Best SLIMElasticNet params: {'l1_ratio': 0.42825676292409526, 'alpha': 0.0010061033871087777, 'positive_only': False, 'topK': 670}
Best RP3beta params: {'alpha': 1.5509591048362328, 'beta': 0.30751121269314674, 'topK': 39, 'implicit': True}
FOLD 1/5


100%|█████████▉| 6968/6969 [01:24<00:00, 82.86it/s] 


RP3betaRecommender: Similarity column 6969 (100.0%), 4103.90 column/sec. Elapsed time 1.70 sec
FOLD 2/5


100%|█████████▉| 6968/6969 [01:25<00:00, 81.86it/s] 


RP3betaRecommender: Similarity column 6969 (100.0%), 4054.56 column/sec. Elapsed time 1.72 sec
FOLD 3/5


...

RP3betaRecommender: Similarity column 6969 (100.0%), 4014.78 column/sec. Elapsed time 1.74 sec
FOLD 4/5


100%|█████████▉| 6968/6969 [01:25<00:00, 81.31it/s] 


RP3betaRecommender: Similarity column 6969 (100.0%), 4011.50 column/sec. Elapsed time 1.74 sec
FOLD 5/5


100%|█████████▉| 6968/6969 [01:26<00:00, 80.68it/s] 


RP3betaRecommender: Similarity column 6969 (100.0%), 4029.35 column/sec. Elapsed time 1.73 sec


## **Mix similarity and tune weights**
W_new = (1 - α) ⋅ RP3
        + α ⋅ (SLIM_pos
               − β ⋅ SLIM_neg)

In [10]:
from Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender
import optuna
from Challenge.hyper_tuning import ModelOptimizer

optimizer = ModelOptimizer("SLIM+RP3beta")

STUDY_NAME = "SLIM_RP3beta_Recommender_1"

In [11]:
def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    params = {
        "alpha": optuna_trial.suggest_float("alpha", 0.0, 1.0),
        "beta": optuna_trial.suggest_float("beta", 0.0, 1.0)
    }
    
    validation_scores = []
    for fold_idx, (URM_train, URM_validation) in enumerate(folds):
        # Train the recommender
        rp3beta = RP3beta[fold_idx]
        slim_positive = SLIM_pos[fold_idx]
        slim_negative = SLIM_neg[fold_idx]
        
        new_similarity = (1 - params["alpha"]) * rp3beta + params["alpha"] * (slim_positive - params["beta"] * slim_negative)

        recommender = ItemKNNCustomSimilarityRecommender(URM_train)
        recommender.fit(new_similarity)
        
        # Evaluate
        evaluator = evaluators[fold_idx]
        score = evaluator.evaluateRecommender(recommender)[0]['RECALL'].iloc[0]
        validation_scores.append(score)
        
        # Show fold result
        print(f"  Fold {fold_idx+1}/{len(folds)} - Score: {score}")

        # Report intermediate result to Optuna
        optuna_trial.report(score, fold_idx)

        # Ask Optuna to prune if performance is poor
        if optuna_trial.should_prune():
            # Return the average score so far instead of raising TrialPruned,
            # which is a common workaround for WilcoxonPruner.
            return np.mean(validation_scores)
        
    # Log folds performance
    optimizer.log_folds(validation_scores, params)

    # Return the mean CV score for the fully completed trial
    return np.mean(validation_scores)

In [12]:
optuna_study = optimizer.create_and_optimize_study(
    study_name=STUDY_NAME,
    objective_function=objective_function,
    n_trials=100
)

[I 2025-11-21 21:55:55,367] A new study created in RDB with name: SLIM_RP3beta_Recommender_1


  0%|          | 0/100 [00:00<?, ?it/s]

EvaluatorHoldout: Processed 27055 (100.0%) in 7.65 sec. Users per second: 3537
  Fold 1/5 - Score: 0.2671636741407766
EvaluatorHoldout: Processed 27057 (100.0%) in 7.42 sec. Users per second: 3647
  Fold 2/5 - Score: 0.26743714080062525
EvaluatorHoldout: Processed 27050 (100.0%) in 7.36 sec. Users per second: 3676
  Fold 3/5 - Score: 0.2717904385274616
EvaluatorHoldout: Processed 27066 (100.0%) in 7.36 sec. Users per second: 3676
  Fold 4/5 - Score: 0.2665490931347514
EvaluatorHoldout: Processed 27065 (100.0%) in 7.48 sec. Users per second: 3620
  Fold 5/5 - Score: 0.2706974232291586
[I 2025-11-21 21:56:32,891] Trial 0 finished with value: 0.2687275539665547 and parameters: {'alpha': 0.4992197683235964, 'beta': 0.9873416742961064}. Best is trial 0 with value: 0.2687275539665547.
EvaluatorHoldout: Processed 27055 (100.0%) in 7.46 sec. Users per second: 3628
  Fold 1/5 - Score: 0.2746092452767785
EvaluatorHoldout: Processed 27057 (100.0%) in 7.39 sec. Users per second: 3659
  Fold 2/5 - 

In [13]:
optuna.visualization.plot_optimization_history(optuna_study)

In [14]:
optuna.visualization.plot_param_importances(optuna_study)

In [15]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

Best Value: 0.28921885019659144

Best Params: {'alpha': 0.9126001919241279, 'beta': 0.14844700846679323}

## **Submission**

In [ ]:
# Train SLIM ElasticNet and RP3beta models on all data
URM_train, URM_val = folds[0]

slim_all = MultiThreadSLIM_SLIMElasticNetRecommender(URM_train+URM_val)
slim_all.fit(**best_slim_params, workers=8)

rp3beta_all = RP3betaRecommender(URM_train+URM_val)
rp3beta_all.fit(**best_rp3_params)

100%|█████████▉| 6968/6969 [02:16<00:00, 50.93it/s]


RP3betaRecommender: Similarity column 6969 (100.0%), 3124.04 column/sec. Elapsed time 2.23 sec


In [20]:
rp3beta = rp3beta_all.W_sparse.copy()
slim_positive = slim_all.W_sparse.copy()
slim_negative = slim_all.W_sparse.copy()
slim_positive.data[slim_positive.data < 0] = 0
slim_negative.data[slim_negative.data > 0] = 0
slim_negative.data = np.abs(slim_negative.data)
# Normalize
assert slim_positive.min() == slim_negative.min() == 0
assert slim_positive.max() > 0 and slim_negative.max() > 0
slim_positive_max = slim_positive.max()
slim_positive.data /= slim_positive_max
slim_negative_max = slim_negative.max()
slim_negative.data /= slim_negative_max
assert rp3beta.min() == 0
assert rp3beta.max() > 0
rp3beta_max = rp3beta.max()
rp3beta.data /= rp3beta_max

In [21]:
alpha = 0.9126001919241279
beta = 0.14844700846679323

similarity = (1 - alpha) * rp3beta + alpha * (slim_positive - beta * slim_negative)

recommender = ItemKNNCustomSimilarityRecommender(URM_train)
recommender.fit(similarity)

In [22]:
import pandas as pd

# Generate recommendations for the test set
user_ids_test = pd.read_csv(paths.CHALLENGE_USER_IDS_TEST)
ids = user_ids_test["user_id"].values

recommendations = recommender.recommend(ids, cutoff=20)

os.makedirs(paths.SUBMISSIONS, exist_ok=True)
with open(os.path.join(paths.SUBMISSIONS, STUDY_NAME + ".csv"), "w") as f:
    f.write("user_id,item_list\n")
    for user_id, rec_list in zip(ids, recommendations):
        f.write(f"{user_id},{' '.join([str(item) for item in rec_list])}\n")